In [10]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter
from langchain_community.vectorstores import Chroma

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

SYSTEM_PROMPT = """Role:
    คุณเป็นผู้เชี่ยวชาญด้านความมั่นคงปลอดภัยไซเบอร์ และการตรวจสอบข้อเท็จจริงของข้อความ SMS ในประเทศไทย ที่มีประสบการณ์สูงมานานกว่า 50 ปี

Task:
    หน้าที่ของคุณคือวิเคราะห์ข้อความ SMS ที่ได้รับ เพื่อตัดสินอย่างตรงไปตรงมาและเป็นกลางว่าข้อความนี้เป็น "SMS หลอกลวง (Scam)" หรือ "SMS ปกติ (Normal)" 
    คุณต้องไม่อคติว่าข้อความที่ได้รับมาจะเป็น Scam เสมอไป แต่ให้ประเมินจากหลักฐานและบริบทที่มีอยู่จริง

Analysis method:
    1. วิเคราะห์ลักษณะของข้อความ: มองหาจุดที่น่าสงสัย (เช่น ลิงก์แปลกปลอม, การเร่งรัด, ข้อเสนอดีเกินจริง)
    2. ตรวจสอบความถูกต้องของลิงก์หรือโดเมน: เปรียบเทียบลิงก์ในข้อความกับโดเมนอย่างเป็นทางการขององค์กรนั้นๆ
    3. พิจารณาเจตนาของข้อความ: ข้อความมีเจตนาเพื่อหลอกให้คลิก/โอนเงิน หรือเพียงแค่ให้ข้อมูล/ทำการตลาดตามปกติ
    4. ใช้ข้อมูลจาก Context ที่เกี่ยวข้องเพื่อสนับสนุนการวิเคราะห์ของคุณ

Response format:
    ตอบเป็นภาษาไทยเท่านั้น กระชับ เข้าใจง่าย และเป็นกลาง โดยใช้โครงสร้างดังนี้:
    - ผลการวิเคราะห์: [ระบุ "SMS หลอกลวง (Scam)" หรือ "SMS ปกติ (Normal)"]
    - เหตุผลและจุดสังเกต: (อธิบายจุดที่น่าสงสัย หรือ จุดที่ยืนยันว่าเป็นข้อความปกติ อย่างมีเหตุผล)
    - ข้อมูลอ้างอิงจาก Context: (ระบุข้อมูลหรือนโยบายที่เกี่ยวข้อง หากไม่มีก็ไม่ต้องระบุ)
    - สรุป: (สรุปเหตุผลหลักสั้นๆ ใน 1-2 ประโยค)

Important:
    - คห้ามตีความเข้าข้างว่าเป็น Scam หากข้อความนั้นเป็นข้อความแจ้งเตือนทั่วไป (เช่น OTP, โปรโมชัน AIS/TRUE, แจ้งส่งพัสดุ) ที่ไม่มีลิงก์อันตรายหรือการแอบอ้าง
    - ให้ใช้เฉพาะข้อมูลจาก Context ที่ให้มาเท่านั้นในการอ้างอิงนโยบาย=ที่ถูกต้อง ห้ามแต่งนโยบายหรือชื่อโดเมนขึ้นมาเองเด็ดขาด
    - หาก Context ไม่เพียงพอและข้อความไม่มีลักษณะอันตรายชัดเจน ให้วิเคราะห์จากจากลักษณะของข้อความแทน
    """

USER_PROMPT_TEMPLATE = """ข้อความ SMS: {sms_message}
นโยบายอ้างอิง:
{retrieved_context}"""

def analyze_scam_sms(sms_message: str, retriever, api_key: str = None) -> str:

    retrieved_docs = retriever.invoke(sms_message)

    context = "\n\n---\n\n".join([
        f"[ที่มา: {doc.metadata.get('source', 'ระบุแหล่งที่มา')}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    user_prompt = USER_PROMPT_TEMPLATE.format(
        sms_message=sms_message,
        retrieved_context=context
    )

    with OpenRouter(api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ]
        )

    return response.choices[0].message.content


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

persist_directory = "./chroma_db"
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

retriever = vectorstore.as_retriever(
    search_type="similarity",           
    search_kwargs={"k": 3}
)

sms_message = [
    # Scam messages
    "ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz",
    "คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi",
    "KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br ",
    "เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค",
    "รับฟรี 129 บาท ฝากแรกรับ 1599 บาท",
    # Normal messages
    "AIS ยกขบวนมือถือรุ่นเด็ด พร้อมโปรโมชันส่วนลดจัดเต็มในงาน Thailand Mobile Expo ณ ศูนย์การประชุมแห่งชาติสิริกิติ์ ระหว่างวันที่ 8-11 กุมภาพันธ์ 2567 เตรียมมาชอปมือถือใหม่ รับปีมังกร พร้อมลดภาษีได้เต็ม Max (ขึ้นอยู่กับอัตราเงินได้สุทธิที่เสียภาษี) แล้วพบกันภายในงานตั้งแต่ 10.00 น. – 20.00 น.",
    "OTP 930917 (ref. 8697) สำหรับสมัคร myID อย่าบอกรหัสนี้กับบุคคลอื่น",
    "ใจฟู เน็ตฟูลสปีด! เล่นไม่สะดุด 50วัน 199บ. 30วัน ถึง 29 ก.พ.67 คลิก https://dtac.co.th/s/CMTDTE5",
    "[ ⬛️ Express] พัสดุ ⬛️⬛️ จัดส่งสำเร็จแล้ว ตรวจสอบได้ที่เว็บไซต์ ⬛️ Express Thailandหากมีข้อสงสัย กรุณาติดต่อ call center 1470",
    "มีผู้เข้าสู่ระบบธนาคารของคุณจากอุปกรณ์อื่น หากไม่ได้ดำเนินการด้วยตนเอง โปรดติดต่อทันที kasikorn.go-line.cc"
]

print("\n=== การวิเคราะห์ SMS ด้วย AI ===")

for i, msg in enumerate(sms_message):
    print(f"\n{'='*15} วิเคราะห์ข้อความที่ {i+1} {'='*15}")
    print(f" SMS: {msg}")
    
    try:
        result = analyze_scam_sms(sms_message=msg, retriever=retriever, api_key=api_key)
        
        print("---  ผลลัพธ์จาก AI ---")
        print(result)
        
    except Exception as e:
        print(f" เกิดข้อผิดพลาดที่ข้อความ: {e}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3421.48it/s]



=== การวิเคราะห์ SMS ด้วย AI ===

=============== วิเคราะห์ข้อความที่ 1 ===============
 SMS: ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz
---  ผลลัพธ์จาก AI ---
ผลการวิเคราะห์: **SMS หลอกลวง (Scam)**  
เหตุผลและจุดสังเกต:  
- ข้อความแจ้งให้รับวงเงินสูงถึง 50,000 บาทพร้อมลิงก์ [rb.gy/2erksz] โดยไม่ระบุชื่อธนาคารหรือนามสกุลของบริษัทอย่างชัดเจน  
- ลิงก์มีรูปแบบเป็น Shortened URL (rb.gy) ซึ่งเป็นวิธีที่นิยมใช้ในข้อความหลอกลวงเพื่อปกปิดเว็บไซต์จริง  
- เจตนาของข้อความกระตุ้นให้คลิกเพื่อรับสิทธิ์อย่างเร่งด่วน ซึ่งเป็นกลยุทธ์หลอกให้เสียการควบคุมตัดสินใจตามหลักการ Social Engineering  

ข้อมูลอ้างอิงจาก Context: **ไม่มีข้อมูลธนาคารหรือโดเมนใดใน Context ที่ให้การรับรองกับลิงก์ดังกล่าว**  

สรุป: ข้อความนี้มีลักษณะเสี่ยงสูงจาก Shortened URL พร้อมข้อเสนอดีเกินจริง โดยไม่มีข้อมูลยืนยันความถูกต้องตามบริบทที่ให้มา จึงจัดอยู่ในประเภทข้อความหลอกลวง.

=============== วิเคราะห์ข้อความที่ 2 ===============
 SMS: คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดข